In [4]:
import os
import sys
import time
from datetime import datetime
from typing import Optional, List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd # Changed from polars
from binance.client import Client # Assuming you have this import
from loguru import logger

# Your logger setup remains the same
logger.remove()
logger.add(
    sys.stdout,
    format="<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> - <level>{message}</level>",
    level="INFO"
)
logger.add(
    "binance_data_{time:YYYY-MM-DD}.log",
    rotation="1 day",
    retention="7 days",
    level="DEBUG",
    format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {name}:{function}:{line} - {message}"
)

class BinanceDataFetcherPandas: # Renamed class for clarity
    """Fetch OHLCV data from Binance using Pandas for data processing"""

    INTERVAL_MAP = {
        '1m': Client.KLINE_INTERVAL_1MINUTE,
        '3m': Client.KLINE_INTERVAL_3MINUTE,
        '5m': Client.KLINE_INTERVAL_5MINUTE,
        '15m': Client.KLINE_INTERVAL_15MINUTE,
        '30m': Client.KLINE_INTERVAL_30MINUTE,
        '1h': Client.KLINE_INTERVAL_1HOUR,
        '2h': Client.KLINE_INTERVAL_2HOUR,
        '4h': Client.KLINE_INTERVAL_4HOUR,
        '6h': Client.KLINE_INTERVAL_6HOUR,
        '8h': Client.KLINE_INTERVAL_8HOUR,
        '12h': Client.KLINE_INTERVAL_12HOUR,
        '1d': Client.KLINE_INTERVAL_1DAY,
        '3d': Client.KLINE_INTERVAL_3DAY,
        '1w': Client.KLINE_INTERVAL_1WEEK,
        '1M': Client.KLINE_INTERVAL_1MONTH
    }

    def __init__(self, api_key: Optional[str] = None, api_secret: Optional[str] = None):
        """Initialize with API credentials"""
        logger.info("Initializing BinanceDataFetcherPandas")

        self.api_key = api_key or os.getenv('BINANCE_API_KEY')
        self.api_secret = api_secret or os.getenv('BINANCE_API_SECRET')

        if not self.api_key or not self.api_secret:
            logger.error("Binance API credentials not found in environment variables")
            raise ValueError("Binance API credentials not found")

        logger.debug(f"API Key found: {self.api_key[:8]}...")

        try:
            self.client = Client(self.api_key, self.api_secret)
            logger.success("Binance client initialized successfully")
        except Exception as e:
            logger.exception(f"Failed to initialize Binance client: {e}")
            raise

        self._test_connection()

    def _test_connection(self):
        """Test API connection"""
        logger.info("Testing Binance API connection...")
        try:
            server_time = self.client.get_server_time()
            server_datetime = datetime.fromtimestamp(server_time['serverTime']/1000)
            logger.success(f"Connected to Binance | Server time: {server_datetime}")
        except Exception as e:
            logger.error(f"Failed to connect to Binance API: {e}")
            raise ConnectionError(f"Failed to connect to Binance: {e}")

    def _process_klines_pandas(self, klines: list, ticker: str) -> pd.DataFrame:
        """Process raw klines data into a clean pandas DataFrame with DatetimeIndex and OHLCV columns."""
        # Standard column names from Binance API for klines
        # [0: Open time, 1: Open, 2: High, 3: Low, 4: Close, 5: Volume, ...]
        column_names = [
            'Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume',
            'Close_Time', 'Quote_Asset_Volume', 'Number_of_Trades',
            'Taker_Buy_Base_Asset_Volume', 'Taker_Buy_Quote_Asset_Volume', 'Ignore'
        ]
        df = pd.DataFrame(klines, columns=column_names)

        if df.empty:
            logger.warning(f"No kline data to process for {ticker}, DataFrame is empty.")
            return pd.DataFrame() # Return empty DataFrame if no klines

        try:
            # Convert timestamp to datetime and set as index
            df['Date'] = pd.to_datetime(df['Timestamp'], unit='ms')
            df = df.set_index('Date')

            # Select and convert OHLCV columns to numeric
            ohlcv_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
            for col in ohlcv_cols:
                df[col] = pd.to_numeric(df[col], errors='raise') # Raise error if conversion fails

            # Keep only the required OHLCV columns
            df = df[ohlcv_cols]
            
            # Ensure standard column names (Pandas is case-sensitive, yfinance often uses capitalized)
            # Our column_names are already capitalized, so this should be fine.

        except Exception as e:
            logger.exception(f"Failed to process klines for {ticker} into pandas DataFrame: {e}")
            return pd.DataFrame() # Return empty DataFrame on processing error

        return df

    @logger.catch # Loguru's catch decorator handles exceptions in the decorated function
    def fetch_ohlcv(self,
                    ticker: str,
                    start_date: str = '2025-01-01',
                    end_date: str = '2025-05-29', # Default to a date in the past if not specified by user
                    interval: str = '1d',
                    rate_limit_delay: float = 0.1) -> pd.DataFrame:
        """
        Fetch OHLCV data for a single ticker, returns a pandas DataFrame.
        """
        logger.info(f"Fetching data | Ticker: {ticker} | Period: {start_date} to {end_date} | Interval: {interval}")

        interval_constant = self.INTERVAL_MAP.get(interval.lower())
        if not interval_constant:
            logger.error(f"Invalid interval: {interval}")
            # Consider raising ValueError here or returning empty df if that's preferred for batch jobs
            raise ValueError(f"Invalid interval: {interval}. Valid intervals: {list(self.INTERVAL_MAP.keys())}")

        time.sleep(rate_limit_delay) # Respect rate limits

        try:
            klines = self.client.get_historical_klines(
                symbol=ticker,
                interval=interval_constant,
                start_str=start_date,
                end_str=end_date
            )
            if not klines: # No data returned from API
                logger.warning(f"No data returned from API for {ticker} (Period: {start_date} to {end_date}, Interval: {interval})")
                return pd.DataFrame()

            logger.success(f"Retrieved {len(klines)} klines for {ticker}")

        except Exception as e: # Catch potential API errors or other issues
            logger.error(f"Failed to fetch klines from Binance API for {ticker}: {e}")
            return pd.DataFrame() # Return empty DataFrame on API failure

        # Process klines into pandas DataFrame
        df = self._process_klines_pandas(klines, ticker)
        return df


    def fetch_multiple_tickers(self,
                               tickers: List[str],
                               start_date: str = '2025-01-01',
                               end_date: str = '2025-05-29',
                               interval: str = '1d',
                               max_workers: int = 5,
                               rate_limit_delay: float = 0.1) -> Dict[str, pd.DataFrame]:
        """
        Fetch OHLCV data for multiple tickers concurrently.
        Returns a dictionary with ticker symbols as keys and pandas DataFrames as values.
        Each DataFrame has a DatetimeIndex and OHLCV columns.
        """
        logger.info(f"Fetching data for {len(tickers)} tickers using up to {max_workers} workers")

        results_dict: Dict[str, pd.DataFrame] = {}
        failed_tickers: List[str] = []

        # Determine if sequential or concurrent fetching
        if len(tickers) <= 3 or max_workers <= 1:
            logger.info("Using sequential fetching.")
            for ticker in tickers:
                df = self.fetch_ohlcv(ticker, start_date, end_date, interval, rate_limit_delay)
                if not df.empty:
                    results_dict[ticker] = df
                else:
                    # fetch_ohlcv logs details if data is not found or errors occur
                    failed_tickers.append(ticker)
        else:
            logger.info(f"Using concurrent fetching with {max_workers} workers.")
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                future_to_ticker = {
                    executor.submit(
                        self.fetch_ohlcv,
                        ticker, start_date, end_date, interval, rate_limit_delay
                    ): ticker
                    for ticker in tickers
                }

                for future in as_completed(future_to_ticker):
                    ticker = future_to_ticker[future]
                    try:
                        df = future.result() # result() will re-raise exceptions from fetch_ohlcv
                        if not df.empty:
                            results_dict[ticker] = df
                        else:
                            failed_tickers.append(ticker) # No data, or empty df returned by fetch_ohlcv
                    except Exception as e:
                        # This catches exceptions not handled by @logger.catch in fetch_ohlcv
                        # or if fetch_ohlcv re-raises an exception.
                        logger.error(f"Error processing future for ticker {ticker}: {e}")
                        failed_tickers.append(ticker)
        
        successful_fetches = len(results_dict)
        logger.info(f"Successfully fetched and processed data for {successful_fetches} out of {len(tickers)} tickers.")
        if failed_tickers:
            logger.warning(f"Failed to fetch or no data returned for {len(failed_tickers)} tickers: {failed_tickers}")

        return results_dict

    def get_multiindex_ohlcv_dataframe(self, data_dict: Dict[str, pd.DataFrame]) -> pd.DataFrame:
        """
        Combines a dictionary of ticker DataFrames (DatetimeIndex, OHLCV columns)
        into a single pandas DataFrame with a MultiIndex column structure (Metric, Ticker).
        """
        if not data_dict:
            logger.warning("Input data dictionary is empty. Returning empty DataFrame.")
            return pd.DataFrame()

        # Filter out any DataFrames that might be empty, though fetch_multiple_tickers aims to avoid this.
        valid_data_dict = {ticker: df for ticker, df in data_dict.items() if not df.empty}

        if not valid_data_dict:
            logger.warning("All DataFrames in the provided dictionary are empty. Returning empty DataFrame.")
            return pd.DataFrame()

        logger.info(f"Combining data for {len(valid_data_dict)} tickers into a MultiIndex DataFrame.")
        
        try:
            # Concatenate along columns (axis=1). Dictionary keys become the outer level of columns.
            # At this point, columns are (Ticker, Metric), e.g., ('BTC-USD', 'Open')
            combined_df = pd.concat(valid_data_dict, axis=1)

            if combined_df.empty: # Should not happen if valid_data_dict was not empty
                logger.warning("Concatenation resulted in an empty DataFrame.")
                return pd.DataFrame()

            # Swap the levels of the column MultiIndex to (Metric, Ticker)
            # e.g., ('Open', 'BTC-USD')
            combined_df = combined_df.swaplevel(0, 1, axis=1)

            # Sort the columns to group by metric, then by ticker, for a consistent order
            # e.g., ('Close', 'BTC-USD'), ('Close', 'ETH-USD'), ('High', 'BTC-USD'), ...
            combined_df = combined_df.sort_index(axis=1, level=[0, 1])

            # Optional: Assign names to the levels of the MultiIndex
            combined_df.columns.names = ['Metric', 'Ticker']

            logger.success(f"Successfully created MultiIndex DataFrame. Shape: {combined_df.shape}")
            return combined_df
        except Exception as e:
            logger.exception(f"Failed to create MultiIndex DataFrame: {e}")
            return pd.DataFrame()

In [5]:
api_key = "nJQUdcloThL02ov1FCY1WLwujVf5KCsXJBqkKWsLlfirfqaSKZ74tdhmxy9qavzQ"
api_secret = "vqL67TezUN5Vbn0DJgy2yFDlpG4JYPMH8WRUIfLzSZEJKql4Q3kXopdtWZH2OUqk"

loader = BinanceDataFetcherPandas(api_key=api_key, api_secret=api_secret)
df = loader.fetch_multiple_tickers(tickers=["BTCUSDT", "ETHUSDT"], interval="1h")

2025-06-01 10:08:00 | INFO     | __main__:__init__:50 - Initializing BinanceDataFetcherPandas
2025-06-01 10:08:01 | SUCCESS  | __main__:__init__:63 - Binance client initialized successfully
2025-06-01 10:08:01 | INFO     | __main__:_test_connection:72 - Testing Binance API connection...
2025-06-01 10:08:01 | SUCCESS  | __main__:_test_connection:76 - Connected to Binance | Server time: 2025-06-01 10:08:00.620000
2025-06-01 10:08:01 | INFO     | __main__:fetch_multiple_tickers:172 - Fetching data for 2 tickers using up to 5 workers
2025-06-01 10:08:01 | INFO     | __main__:fetch_multiple_tickers:179 - Using sequential fetching.
2025-06-01 10:08:01 | INFO     | __main__:fetch_ohlcv:128 - Fetching data | Ticker: BTCUSDT | Period: 2025-01-01 to 2025-05-29 | Interval: 1h
2025-06-01 10:08:04 | SUCCESS  | __main__:fetch_ohlcv:149 - Retrieved 3553 klines for BTCUSDT
2025-06-01 10:08:04 | INFO     | __main__:fetch_ohlcv:128 - Fetching data | Ticker: ETHUSDT | Period: 2025-01-01 to 2025-05-29 | I

In [7]:
multiindex_df = loader.get_multiindex_ohlcv_dataframe(df)
multiindex_df.head()

2025-06-01 10:09:37 | INFO     | __main__:get_multiindex_ohlcv_dataframe:235 - Combining data for 2 tickers into a MultiIndex DataFrame.
2025-06-01 10:09:37 | SUCCESS  | __main__:get_multiindex_ohlcv_dataframe:257 - Successfully created MultiIndex DataFrame. Shape: (3553, 10)


Metric                  Close               High                Low           \
Ticker                BTCUSDT  ETHUSDT   BTCUSDT  ETHUSDT   BTCUSDT  ETHUSDT   
Date                                                                           
2025-01-01 00:00:00  94401.14  3363.70  94509.42  3365.71  93489.03  3335.84   
2025-01-01 01:00:00  93607.74  3346.54  94408.72  3366.40  93578.77  3342.67   
2025-01-01 02:00:00  94098.91  3362.61  94105.12  3368.42  93594.56  3346.35   
2025-01-01 03:00:00  93838.04  3355.20  94098.91  3363.72  93728.22  3351.00   
2025-01-01 04:00:00  93553.91  3341.14  93838.04  3356.61  93500.00  3339.45   

Metric                   Open              Volume             
Ticker                BTCUSDT  ETHUSDT    BTCUSDT    ETHUSDT  
Date                                                          
2025-01-01 00:00:00  93576.00  3337.78  755.99010  8894.4444  
2025-01-01 01:00:00  94401.13  3363.69  586.53456  6138.1376  
2025-01-01 02:00:00  93607.74  3346.54  276.78045  8775.8100  
2025-01-01 03:00:00  94098.90  3362.61  220.99302  4025.4646  
2025-01-01 04:00:00  93838.04  3355.20  279.46909  4626.6506